# Experiments
Notebook per confrontare configurazioni di iperparametri con la **cross-validation**.

- La CV è fatta da `gmnbind.cross_validation.cross_validate`: un modello nuovo per ogni fold, early stopping su `val_loss`,
  e tutte le metriche (RMSE, MAE, Pearson, Spearman, SD, CI) calcolate sul **miglior checkpoint** di ogni fold.
- I fold sono fissi e letti da file (`data/index/splits/v2020_casf_cv5/fold_*`): tutti gli esperimenti usano esattamente gli stessi dati.
  Il test set (CASF-2016) non fa parte della CV.
- Ogni esperimento = una config base (`configs/*.yaml`) + le modifiche definite in questo notebook.
- Log: `lightning_logs/<nome>_cv/fold_i/`, più `cv_folds.csv` e `cv_summary.json` per ogni esperimento.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*torch.jit.script.*is deprecated.*", category=FutureWarning)

import copy
import json
import os
from pathlib import Path

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from gmnbind.cross_validation import cross_validate, load_config

# il notebook gira da notebooks/ -> la radice della repo è la cartella sopra
REPO_ROOT = Path.cwd().parent
assert (REPO_ROOT / 'configs').exists(), f"REPO_ROOT sbagliata: {REPO_ROOT}"

CV_DIR = REPO_ROOT / 'data' / 'index' / 'splits' / 'v2020_casf_cv5'
BASE_CONFIGS = {
    'baseline': REPO_ROOT / 'configs' / 'baseline.yaml',
    'gmn':      REPO_ROOT / 'configs' / 'gmn.yaml',
}
LOG_DIR = REPO_ROOT / 'lightning_logs'

# experiment analysis

In [ ]:
from matplotlib.ticker import MaxNLocator
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import glob

LIGHTNING_LOGS_DIR = str(REPO_ROOT / 'lightning_logs')

def _scan_experiments(logs_dir):
    entries = []
    for exp_path in sorted(glob.glob(os.path.join(logs_dir, '*'))):
        if not os.path.isdir(exp_path):
            continue
        exp_name = os.path.basename(exp_path)
        # train.py runs -> version_*, cross-validation runs -> fold_*
        ver_paths = glob.glob(os.path.join(exp_path, 'version_*')) + glob.glob(os.path.join(exp_path, 'fold_*'))
        for ver_path in sorted(p for p in ver_paths if os.path.isdir(p)):
            ver_name = os.path.basename(ver_path)
            entries.append((f"{exp_name} / {ver_name}", exp_path, ver_path))
    return entries

def _load_metrics(ver_path):
    csv_path = os.path.join(ver_path, 'metrics.csv')
    return pd.read_csv(csv_path) if os.path.exists(csv_path) else None

def _load_hparams(ver_path):
    hp_path = os.path.join(ver_path, 'hparams.yaml')
    if not os.path.exists(hp_path):
        return {}
    with open(hp_path) as f:
        return yaml.safe_load(f) or {}

def _render_dashboard(ver_path, out):
    with out:
        clear_output(wait=True)
        metrics = _load_metrics(ver_path)
        hparams = _load_hparams(ver_path)

        if metrics is None:
            print("metrics.csv non trovato in", ver_path)
            return

        val_col  = metrics['val_loss'].dropna()
        # new runs log 'val_pearson', old runs 'val_pearson_r'
        pear_name = 'val_pearson' if 'val_pearson' in metrics else 'val_pearson_r'
        pear_col = metrics[pear_name].dropna() if pear_name in metrics else pd.Series(dtype=float)

        # LR: cerca la prima colonna che inizia con 'lr-' (lr-AdamW, lr-Adam, ecc.)
        lr_col_name = next((c for c in metrics.columns if c.startswith('lr-')), None)
        lr_data = metrics[['step', lr_col_name]].dropna() if lr_col_name else None

        best_val  = val_col.min()     if len(val_col)  else float('nan')
        best_pear = pear_col.max()    if len(pear_col) else float('nan')
        last_pear = pear_col.iloc[-1] if len(pear_col) else float('nan')
        tot_epochs = int(metrics['epoch'].max()) + 1

        # --- riepilogo testo ---
        print("─" * 60)
        print("  Iperparametri")
        print("─" * 60)
        for k, v in hparams.items():
            print(f"  {k:<28} {v}")
        print()
        print("─" * 60)
        print("  Riepilogo metriche")
        print("─" * 60)
        print(f"  Epoche totali:          {tot_epochs}")
        print(f"  Best val_loss (MSE):    {best_val:.4f}")
        if not pd.isna(best_pear):
            print(f"  Best val_pearson:       {best_pear:.4f}   (finale: {last_pear:.4f})")
        print()

        train_m = metrics[metrics['train_loss'].notna()]
        train_per_epoch = train_m.groupby('epoch')['train_loss'].mean()
        val_m = metrics[metrics['val_loss'].notna()].set_index('epoch')['val_loss']

        has_lr   = lr_data is not None and len(lr_data) > 0
        has_pear = len(pear_col) > 0

        # --- riga superiore: fino a 3 pannelli ---
        n_top = 1 + int(has_lr) + int(has_pear)
        fig_top, axes = plt.subplots(1, n_top, figsize=(5 * n_top, 4))
        axes = list(axes) if n_top > 1 else [axes]

        axes[0].plot(train_per_epoch.index, train_per_epoch.values,
                     color='steelblue', linewidth=1.5, label='train loss')
        axes[0].plot(val_m.index, val_m.values,
                     color='tomato', marker='o', markersize=2, linewidth=1.5, label='val loss')
        axes[0].set_title('Loss (per epoch)')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('MSE Loss')
        axes[0].legend()

        ax_idx = 1

        if has_lr:
            axes[ax_idx].plot(lr_data['step'], lr_data[lr_col_name],
                              color='darkorange', linewidth=1.5)
            axes[ax_idx].set_title(f'Learning Rate ({lr_col_name})')
            axes[ax_idx].set_xlabel('Step')
            axes[ax_idx].set_ylabel('LR')
            axes[ax_idx].ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
            ax_idx += 1

        if has_pear:
            pear_m = metrics[metrics[pear_name].notna()]
            axes[ax_idx].plot(pear_m['epoch'], pear_m[pear_name],
                              color='darkorchid', marker='^', markersize=3, linewidth=1.5)
            axes[ax_idx].set_title('Val Pearson r (per epoch)')
            axes[ax_idx].set_xlabel('Epoch')
            axes[ax_idx].set_ylabel('Pearson r')

        for ax in axes:
            ax.grid(True, alpha=0.3)
        fig_top.tight_layout()
        plt.show()

        # --- riga inferiore: zoom convergenza ---
        cutoff = max(1, int(len(train_per_epoch) * 0.15))
        all_conv = pd.concat([train_per_epoch.iloc[cutoff:], val_m.iloc[cutoff:]])
        y_min = all_conv.min() * 0.97
        y_max = np.percentile(all_conv, 97) * 1.03

        fig_zoom, ax_z = plt.subplots(figsize=(5 * n_top, 4))
        ax_z.plot(train_per_epoch.index, train_per_epoch.values,
                  color='steelblue', linewidth=0.9, alpha=0.85, label='train loss')
        ax_z.plot(val_m.index, val_m.values,
                  color='tomato', linewidth=0.9, marker='o', markersize=1.5, alpha=0.85, label='val loss')
        ax_z.set_ylim(y_min, y_max)
        ax_z.yaxis.set_major_locator(MaxNLocator(nbins=12))
        ax_z.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=20))
        ax_z.set_title('Loss — zoom sulla convergenza')
        ax_z.set_xlabel('Epoch')
        ax_z.set_ylabel('MSE Loss')
        ax_z.legend()
        ax_z.grid(True, alpha=0.35, linewidth=0.6)
        fig_zoom.tight_layout()
        plt.show()

# --- UI ---
experiments = _scan_experiments(LIGHTNING_LOGS_DIR)

if not experiments:
    print(f"Nessun esperimento trovato in '{LIGHTNING_LOGS_DIR}/'.")
else:
    labels = [e[0] for e in experiments]
    dd = widgets.Dropdown(
        options=labels,
        description='Esperimento:',
        layout=widgets.Layout(width='420px'),
        style={'description_width': 'initial'},
    )
    out = widgets.Output()

    def _on_change(change):
        if change['name'] == 'value':
            idx = labels.index(change['new'])
            _render_dashboard(experiments[idx][2], out)

    dd.observe(_on_change, names='value')
    display(dd, out)
    _render_dashboard(experiments[0][2], out)

## Configurazioni esperimenti
Ogni esperimento parte dalla config base del modello (`configs/baseline.yaml` o `configs/gmn.yaml`)
e sovrascrive solo i parametri indicati. Il nome diventa `model.experiment_name` e la cartella dei log.

In [ ]:
def make_config(name, model_type, training=None, **model_overrides):
    """Base config of model_type + overrides of the `model` (and optionally `training`) block."""
    cfg = copy.deepcopy(load_config(BASE_CONFIGS[model_type]))
    cfg['model'].update(model_overrides)
    cfg['model']['model_type'] = model_type
    cfg['model']['experiment_name'] = name
    cfg['training'].update(training or {})
    return cfg

# Parametri condivisi tra tutte le coppie
TRAINING   = {'epochs': 2, 'patience': 15, 'batch_size': 16}
TOKEN_DIM  = 32
NODE_DIMS  = [32, 16]
EDGE_DIMS  = [32, 16]
NUM_LAYERS = 1

NO_REG = dict(token_dim=TOKEN_DIM, node_embedding_dims=NODE_DIMS, edge_embedding_dims=EDGE_DIMS,
              num_layers=NUM_LAYERS, lr=1e-3, max_lr=1e-3, min_lr=1e-3, warmup_percentage=0.0,
              l2_norm=0.0, dropout_p=0.0, normalization=False, use_dynamic_schedule=False)

REG = {**NO_REG, 'l2_norm': 0.01, 'dropout_p': 0.1, 'normalization': True}

DYN_SCHED = {**REG, 'token_dim': 64, 'num_layers': 2, 'min_lr': 1e-6, 'warmup_percentage': 0.075,
             'l2_norm': 0.0006, 'dropout_p': 0.12, 'use_dynamic_schedule': True}

EXPERIMENTS = [
    make_config('baseline_no_reg',    'baseline', TRAINING, **NO_REG),
    make_config('gmn_no_reg',         'gmn',      TRAINING, **NO_REG),
    make_config('baseline_reg',       'baseline', TRAINING, **REG),
    make_config('gmn_reg',            'gmn',      TRAINING, **REG),
    make_config('baseline_dyn_sched', 'baseline', TRAINING, **DYN_SCHED),
    make_config('gmn_dyn_sched',      'gmn',      TRAINING, **DYN_SCHED),
]
print([c['model']['experiment_name'] for c in EXPERIMENTS])

## Run
Lancia la CV per ogni esperimento. I risultati di ogni fold vengono salvati man mano,
quindi la cella si può interrompere e i risultati già calcolati restano su disco.

In [ ]:
cv_results = {}
for cfg in EXPERIMENTS:
    name = cfg['model']['experiment_name']
    folds, summary = cross_validate(cfg, cv_dir=CV_DIR)
    cv_results[name] = {'folds': folds, 'summary': summary}

## Risultati
Legge i `cv_summary.json` dal disco: funziona anche senza rilanciare la CV.

In [ ]:
METRICS = ['val_loss', 'val_rmse', 'val_mae', 'val_pearson', 'val_spearman', 'val_sd', 'val_ci']

rows = []
for cfg in EXPERIMENTS:
    name = cfg['model']['experiment_name']
    path = LOG_DIR / f"{name}_cv" / 'cv_summary.json'
    if not path.exists():
        print(f"Nessun cv_summary.json per {name}. Salto...")
        continue
    s = json.loads(path.read_text())
    m = cfg['model']
    row = {'name': name, 'model': s['model_type'], 'n_folds': s['n_folds']}
    for k in METRICS:
        if k in s['metrics']:
            row[f'{k}_mean'] = round(s['metrics'][k]['mean'], 4)
            row[f'{k}_std']  = round(s['metrics'][k]['std'], 4)
    row.update({'token_dim': m['token_dim'], 'node_emb': m['node_embedding_dims'][-1],
                'num_layers': m['num_layers'], 'normalization': m['normalization'],
                'dropout_p': m['dropout_p'], 'l2_norm': m['l2_norm'],
                'dynamic_schedule': m['use_dynamic_schedule'], 'max_lr': m['max_lr'],
                'batch_size': cfg['training']['batch_size']})
    rows.append(row)

df_summary = pd.DataFrame(rows).set_index('name').sort_values('val_loss_mean')
df_summary.to_csv(LOG_DIR / 'cv_comparison.csv')
print(f"Summary salvata in {LOG_DIR / 'cv_comparison.csv'}")
df_summary

In [ ]:
# Baseline vs GMN per configurazione: MSE e Pearson (mean ± std sui fold)
configs = ['no_reg', 'reg', 'dyn_sched']
xtick_labels = ['Standard', 'Regularized', 'Dynamic-Large']
models = ['baseline', 'gmn']
panels = [('val_loss', 'Val Loss MSE', 'MSE', ['#1f77b4', '#ff7f0e']),
          ('val_pearson', 'Val Pearson r', 'Pearson r', ['#9467bd', '#2ca02c'])]

x = np.arange(len(configs))
width = 0.35
err_style = {'elinewidth': 1.2, 'capthick': 1.2, 'alpha': 0.6}
fig, axes = plt.subplots(1, len(panels), figsize=(6 * len(panels), 5))

for ax, (metric, title, ylabel, colors) in zip(axes, panels):
    all_low, all_high = [], []
    for j, (m, color) in enumerate(zip(models, colors)):
        means = [df_summary.loc[f'{m}_{c}', f'{metric}_mean'] if f'{m}_{c}' in df_summary.index else np.nan for c in configs]
        stds  = [df_summary.loc[f'{m}_{c}', f'{metric}_std']  if f'{m}_{c}' in df_summary.index else np.nan for c in configs]
        ax.bar(x + (j - 0.5) * width, means, width, yerr=stds, label=m.capitalize() if m == 'baseline' else 'GMN',
               color=color, capsize=5, alpha=0.85, ecolor='gray', error_kw=err_style)
        all_low += list(np.array(means) - np.array(stds)); all_high += list(np.array(means) + np.array(stds))
    ax.set_title(title); ax.set_ylabel(ylabel)
    ax.set_xticks(x); ax.set_xticklabels(xtick_labels, rotation=30, ha='right', fontsize=9)
    ax.legend(); ax.grid(True, axis='y', alpha=0.3)
    ax.set_ylim([np.nanmin(all_low) * 0.95, np.nanmax(all_high) * 1.05])

plt.suptitle('Cross-Validation Results (mean ± std)', fontsize=14, y=1.05)
plt.tight_layout()
plt.savefig(LOG_DIR / 'cv_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# LR tuner

In [ ]:
from pytorch_lightning.tuner import Tuner
import pytorch_lightning as pl
from gmnbind.data.datamodule import ComplexDataModule
from gmnbind.cross_validation import build_model

tuner_cfg = load_config(BASE_CONFIGS['gmn'])
data_module = ComplexDataModule(
    processed_dir=REPO_ROOT / tuner_cfg['data']['processed_dir'],
    split_dir=CV_DIR / 'fold_0',          # any training split works for the LR finder
    batch_size=16,
)
model = build_model(tuner_cfg['model'])

trainer = pl.Trainer(max_epochs=tuner_cfg['training']['epochs'], precision='bf16-mixed')

print("Avvio la scansione del Learning Rate...")
tuner = Tuner(trainer)
lr_finder = tuner.lr_find(
    model,
    datamodule=data_module,
    num_training=200,
    min_lr=1e-8,
    max_lr=1
)

In [ ]:
# stampa il learning rate suggerito
new_lr = lr_finder.suggestion()
print(f"LR suggerito: {new_lr:.2e}")
fig = lr_finder.plot(suggest=True)
fig.show()